# La Coipa Data Review

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# FILE SETUP
# =============================================================================
file_path = Path("leachit.xlsx")
assert file_path.exists(), f"File not found: {file_path}"
print(f"Reading workbook: {file_path.resolve()}")

# Optional constants
tank_count = 8
tank_vol = 2987  # m3
total_circuit_volume = tank_count * tank_vol
print(f"Total volume of all tanks: {total_circuit_volume} m3")

random_seed = 13
np.random.seed(random_seed)

# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================
op_data_cols = [
    "date",
    "throughput_tpd",
    "au_feed_gpt",
    "ag_feed_gpt",
    "cu_feed_ppm",
    "au_tail_gpt",
    "ag_tail_gpt",
    "nacn_consumption_tpd",
    "specific_nacn_kgpt",
    "tailings_moisture_pct",
    "do_ag1",
    "do_ag2",
    "do_ag3",
    "do_ag4",
    "do_ag5",
    "do_ag6",
    "do_ag7",
    "do_ag8",
]

leach_2025_cols = [
    "date",
    "time",
    "au_ppm_tk_1_e",
    "au_ppm_tk_1_s",
    "au_ppm_tk_6_s",
    "au_ppm_tk_8_s",
    "ag_ppm_tk_1_e",
    "ag_ppm_tk_1_s",
    "ag_ppm_tk_6_s",
    "ag_ppm_tk_8_s",
    "cu_ppm_tk_1_e",
    "cu_ppm_tk_1_s",
    "cu_ppm_tk_6_s",
    "cu_ppm_tk_8_s",
    "zn_ppm_tk_1_e",
    "zn_ppm_tk_1_s",
    "zn_ppm_tk_6_s",
    "zn_ppm_tk_8_s",
    "ph_tk_1_e",
    "ph_tk_1_s",
    "ph_tk_6_s",
    "ph_tk_8_s",
    "free_cn_ppm_tk_1_e",
    "free_cn_ppm_tk_1_s",
    "free_cn_ppm_tk_6_s",
    "free_cn_ppm_tk_8_s",
    "wad_gpl_tk_1_e",
    "wad_gpl_tk_1_s",
    "wad_gpl_tk_6_s",
    "wad_gpl_tk_8_s",
]

leach_2026_cols = [
    "date",
    "time",
    "au_ppm_tk_1_e",
    "au_ppm_tk_1_s",
    "au_ppm_tk_6_s",
    "au_ppm_tk_8_s",
    "ag_ppm_tk_1_e",
    "ag_ppm_tk_1_s",
    "ag_ppm_tk_6_s",
    "ag_ppm_tk_8_s",
    "cu_ppm_tk_1_e",
    "cu_ppm_tk_1_s",
    "cu_ppm_tk_6_s",
    "cu_ppm_tk_8_s",
    "zn_ppm_tk_1_e",
    "zn_ppm_tk_1_s",
    "zn_ppm_tk_6_s",
    "zn_ppm_tk_8_s",
    "pb_ppm_tk_1_e",
    "pb_ppm_tk_8_s",
    "ph_tk_1_e",
    "ph_tk_1_s",
    "ph_tk_6_s",
    "ph_tk_8_s",
    "free_cn_ppm_tk_1_e",
    "free_cn_ppm_tk_1_s",
    "free_cn_ppm_tk_6_s",
    "free_cn_ppm_tk_8_s",
    "wad_gpl_tk_1_e",
    "wad_gpl_tk_1_s",
    "wad_gpl_tk_6_s",
    "wad_gpl_tk_8_s",
]

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def coerce_numeric(df: pd.DataFrame, exclude=None) -> pd.DataFrame:
    """Convert all columns except those excluded to numeric where possible."""
    exclude = set(exclude or [])
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def clean_operational_data(file_path: Path) -> pd.DataFrame:
    """Read and clean the Operational Data sheet."""
    df = pd.read_excel(
        file_path,
        sheet_name="Operational Data",
        skiprows=4,
        usecols="A:R",
        header=None,
    )

    df.columns = op_data_cols

    # Date handling
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

    # Drop empty rows / rows with no date
    df = df.dropna(how="all")
    df = df[df["date"].notna()].copy()

    # Numeric conversion
    df = coerce_numeric(df, exclude=["date"])

    # If there are accidental duplicate dates, average them
    numeric_cols = [c for c in df.columns if c != "date"]
    df = (
        df.groupby("date", as_index=False)[numeric_cols]
        .mean()
        .sort_values("date")
        .reset_index(drop=True)
    )

    return df


def clean_leach_sheet(
    file_path: Path,
    sheet_name: str,
    col_names: list[str],
    usecols: str,
) -> pd.DataFrame:
    """
    Read a leach sheet with multiple measurements per day and aggregate to daily mean.
    Leach sheets have 2 header rows, then data begins.
    """
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        skiprows=2,
        usecols=usecols,
        header=None,
    )

    df.columns = col_names

    # Remove fully empty rows
    df = df.dropna(how="all").copy()

    # Dates are only populated on first row for each day in these sheets
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date"] = df["date"].ffill().dt.normalize()

    # Time is optional for daily averaging, but keep it tidy
    df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.time

    # Remove rows still lacking a valid date
    df = df[df["date"].notna()].copy()

    # Numeric conversion
    df = coerce_numeric(df, exclude=["date", "time"])

    # Average all measurements for each day
    value_cols = [c for c in df.columns if c not in ["date", "time"]]
    df_daily = (
        df.groupby("date", as_index=False)[value_cols]
        .mean()
        .sort_values("date")
        .reset_index(drop=True)
    )

    return df_daily


# =============================================================================
# READ + CLEAN
# =============================================================================
df_op_data = clean_operational_data(file_path)

df_leach_2025_daily = clean_leach_sheet(
    file_path=file_path,
    sheet_name="Leaching 2025",
    col_names=leach_2025_cols,
    usecols="A:AD",
)

df_leach_2026_daily = clean_leach_sheet(
    file_path=file_path,
    sheet_name="Leaching 2026",
    col_names=leach_2026_cols,
    usecols="A:AF",
)

# =============================================================================
# COMBINE LEACH SHEETS
# =============================================================================
# Concatenate and re-average by date in case there is any overlap
df_leach_daily = pd.concat(
    [df_leach_2025_daily, df_leach_2026_daily],
    ignore_index=True,
    sort=False,
)

leach_value_cols = [c for c in df_leach_daily.columns if c != "date"]
df_leach_daily = (
    df_leach_daily.groupby("date", as_index=False)[leach_value_cols]
    .mean()
    .sort_values("date")
    .reset_index(drop=True)
)

# =============================================================================
# MERGE TO ONE ROW PER DAY
# =============================================================================
df_daily = (
    df_op_data.merge(df_leach_daily, on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

# =============================================================================
# DERIVED FIELDS
# =============================================================================
# Keep copper in original ppm, but add pct as well if useful
df_daily["cu_feed_pct"] = df_daily["cu_feed_ppm"] / 10000

# Gold recovery
df_daily["recovery_au_pct"] = np.where(
    df_daily["au_feed_gpt"] > 0,
    ((df_daily["au_feed_gpt"] - df_daily["au_tail_gpt"]) / df_daily["au_feed_gpt"]) * 100,
    np.nan,
)

# Cyanide speciation - how much cyanide is complexed vs free in solution
df_daily["cn_complexed_gpl"] = df_daily["wad_gpl_tk_8_s"] - (df_daily["free_cn_ppm_tk_8_s"] / 1000)

# Average DO across operational analyser points
do_cols = [c for c in df_daily.columns if c.startswith("do_ag")]
df_daily["do_avg"] = df_daily[do_cols].mean(axis=1)

# =============================================================================
# QA CHECKS
# =============================================================================
print("\nShapes")
print("Operational daily:", df_op_data.shape)
print("Leach 2025 daily:", df_leach_2025_daily.shape)
print("Leach 2026 daily:", df_leach_2026_daily.shape)
print("Combined leach daily:", df_leach_daily.shape)
print("Final merged daily:", df_daily.shape)

print("\nDate range")
print("Operational:", df_op_data["date"].min(), "to", df_op_data["date"].max())
print("Leach daily:", df_leach_daily["date"].min(), "to", df_leach_daily["date"].max())
print("Merged:", df_daily["date"].min(), "to", df_daily["date"].max())

print("\nMissing values (top 25 columns)")
print(df_daily.isna().sum().sort_values(ascending=False).head(25))

print("\nPreview")
print(df_daily.head())

print("\nSummary")
print(df_daily.describe(include="all").T.head(40))




## Convert tank CN data

In [ ]:
df = df.copy()

# --- Convert Free CN (ppm → g/L) ---
free_cn_cols = [c for c in df.columns if c.startswith("free_cn_ppm")]

for col in free_cn_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col.replace("ppm", "gpl")] = df[col] / 1000


# --- WAD already in g/L ---
wad_cols = [c for c in df.columns if c.startswith("wad_gpl")]

for col in wad_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## Calculate complexed CN per tank

In [ ]:
complexed_cols = []

for free_col in free_cn_cols:
    base = free_col.replace("free_cn_ppm_", "")
    wad_col = f"wad_gpl_{base}"
    free_gpl_col = free_col.replace("ppm", "gpl")

    if wad_col in df.columns:
        complex_col = f"complexed_cn_gpl_{base}"
        df[complex_col] = (df[wad_col] - df[free_gpl_col]).clip(lower=0)
        complexed_cols.append(complex_col)

## Create system-level CN metrics

In [ ]:
free_gpl_cols = [c for c in df.columns if "free_cn_gpl" in c]
wad_gpl_cols = [c for c in df.columns if c.startswith("wad_gpl_")]

df["free_cn_gpl_avg"] = df[free_gpl_cols].mean(axis=1)
df["wad_cn_gpl_avg"]  = df[wad_gpl_cols].mean(axis=1)
df["complexed_cn_gpl_avg"] = df[complexed_cols].mean(axis=1)

## Cyanide consumption

In [ ]:
df["nacn_consumption_tpd_calc"] = (
    df["specific_nacn_kgpt"] * df["throughput_tpd"] / 1000
)

### Compare with reported

In [ ]:
df["nacn_gap_tpd"] = (
    df["nacn_consumption_tpd"] - df["nacn_consumption_tpd_calc"]
)

df["nacn_gap_pct"] = np.where(
    df["nacn_consumption_tpd_calc"] > 0,
    100 * df["nacn_gap_tpd"] / df["nacn_consumption_tpd_calc"],
    np.nan,
)

## Cyanide complexation index

In [ ]:
df["cn_complexation_index"] = np.where(
    df["wad_cn_gpl_avg"] > 0,
    df["complexed_cn_gpl_avg"] / df["wad_cn_gpl_avg"],
    np.nan,
)

## Copper interaction

In [ ]:
cu_cols = [c for c in df.columns if c.startswith("cu_ppm")]

df["cu_solution_ppm_avg"] = df[cu_cols].mean(axis=1)

### Link copper → cyanide loss

In [ ]:
corr = df[["cu_solution_ppm_avg", "complexed_cn_gpl_avg"]].corr().iloc[0, 1]
print(f"Cu vs CN complexation correlation: {corr:.3f}")

## Operational efficiency metric

In [ ]:
df["nacn_kg_per_au"] = np.where(
    df["au_feed_gpt"] > 0,
    df["specific_nacn_kgpt"] / df["au_feed_gpt"],
    np.nan,
)

## Key diagnostic plots

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# -----------------------------------------------------------------------------
# Ensure dates are datetime
# -----------------------------------------------------------------------------
df = df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# -----------------------------------------------------------------------------
# Styling helpers
# -----------------------------------------------------------------------------
PLOT_BG = "white"
GRID_COLOUR = "rgba(180, 180, 180, 0.25)"
FONT_FAMILY = "Arial"

def apply_clean_layout(fig, title, x_title=None, y_title=None, height=500, width=None):
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        template="plotly_white",
        plot_bgcolor=PLOT_BG,
        paper_bgcolor=PLOT_BG,
        font=dict(family=FONT_FAMILY, size=13),
        hovermode="closest",
        legend=dict(
            title="Click legend to show/hide",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(255,255,255,0.8)"
        ),
        margin=dict(l=70, r=30, t=80, b=60),
        height=height,
        width=width,
    )

    fig.update_xaxes(
        title_text=x_title,
        showgrid=True,
        gridcolor=GRID_COLOUR,
        zeroline=False,
        showline=True,
        linecolor="rgba(0,0,0,0.35)",
        mirror=False
    )

    fig.update_yaxes(
        title_text=y_title,
        showgrid=True,
        gridcolor=GRID_COLOUR,
        zeroline=False,
        showline=True,
        linecolor="rgba(0,0,0,0.35)",
        mirror=False
    )

    return fig


# -----------------------------------------------------------------------------
# 1) NaCN consumption reconciliation
# -----------------------------------------------------------------------------
fig_consumption = go.Figure()

fig_consumption.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["nacn_consumption_tpd"],
        mode="lines+markers",
        name="Reported",
        line=dict(width=3),
        marker=dict(size=5),
        hovertemplate="<b>Reported</b><br>Date: %{x|%Y-%m-%d}<br>NaCN: %{y:.2f} t/d<extra></extra>"
    )
)

fig_consumption.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["nacn_consumption_tpd_calc"],
        mode="lines+markers",
        name="Calculated",
        line=dict(width=3, dash="dash"),
        marker=dict(size=5),
        hovertemplate="<b>Calculated</b><br>Date: %{x|%Y-%m-%d}<br>NaCN: %{y:.2f} t/d<extra></extra>"
    )
)

apply_clean_layout(
    fig_consumption,
    title="NaCN Consumption Reconciliation",
    x_title="Date",
    y_title="NaCN Consumption (t/d)",
    height=520
)

fig_consumption.update_xaxes(
    rangeslider_visible=True,
    tickformat="%b\n%Y"
)

fig_consumption.show()


# -----------------------------------------------------------------------------
# 2) Copper vs cyanide complexation
# -----------------------------------------------------------------------------
scatter_df = df[["cu_solution_ppm_avg", "complexed_cn_gpl_avg", "date"]].dropna().copy()

fig_cu = px.scatter(
    scatter_df,
    x="cu_solution_ppm_avg",
    y="complexed_cn_gpl_avg",
    hover_data={"date": True, "cu_solution_ppm_avg":":.1f", "complexed_cn_gpl_avg":":.2f"},
    opacity=0.8
)

# Restyle the scatter trace
fig_cu.update_traces(
    marker=dict(size=9, line=dict(width=1, color="white")),
    selector=dict(mode="markers"),
    name="Data"
)

# Add simple linear fit if enough data exists
if len(scatter_df) >= 2:
    x = scatter_df["cu_solution_ppm_avg"].to_numpy()
    y = scatter_df["complexed_cn_gpl_avg"].to_numpy()
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = m * x_line + b

    fig_cu.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name="Linear trend",
            line=dict(width=3, dash="dot"),
            hovertemplate="Trend<br>Cu: %{x:.0f} ppm<br>Complexed CN: %{y:.2f} g/L<extra></extra>"
        )
    )

apply_clean_layout(
    fig_cu,
    title="Copper vs Cyanide Complexation",
    x_title="Cu in Solution (ppm)",
    y_title="Complexed CN (g/L)",
    height=520
)

fig_cu.show()


# -----------------------------------------------------------------------------
# 3) Free vs WAD cyanide
# -----------------------------------------------------------------------------
free_wad_df = df[["free_cn_gpl_avg", "wad_cn_gpl_avg", "date"]].dropna().copy()

fig_wad = px.scatter(
    free_wad_df,
    x="free_cn_gpl_avg",
    y="wad_cn_gpl_avg",
    hover_data={"date": True, "free_cn_gpl_avg":":.2f", "wad_cn_gpl_avg":":.2f"},
    opacity=0.8
)

fig_wad.update_traces(
    marker=dict(size=9, line=dict(width=1, color="white")),
    selector=dict(mode="markers"),
    name="Data"
)

# Add 1:1 line
if len(free_wad_df) > 0:
    x_min = free_wad_df["free_cn_gpl_avg"].min()
    x_max = free_wad_df["free_cn_gpl_avg"].max()
    y_min = free_wad_df["wad_cn_gpl_avg"].min()
    y_max = free_wad_df["wad_cn_gpl_avg"].max()
    line_min = min(x_min, y_min)
    line_max = max(x_max, y_max)

    fig_wad.add_trace(
        go.Scatter(
            x=[line_min, line_max],
            y=[line_min, line_max],
            mode="lines",
            name="1:1 line",
            line=dict(width=3, dash="dash"),
            hoverinfo="skip"
        )
    )

apply_clean_layout(
    fig_wad,
    title="Free vs WAD Cyanide",
    x_title="Free CN (g/L)",
    y_title="WAD CN (g/L)",
    height=520
)

fig_wad.show()

In [ ]:
df = df.copy()

df.columns.to_list()

In [ ]:
df.dtypes

In [155]:
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = df.copy()

# use the real date column, not the integer index
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date")

# calculate tank averages
for element in ["au", "ag", "cu", "zn"]:
    tk_cols = [c for c in df.columns if c.startswith(f"{element}_ppm_tk_")]
    df[f"{element}_solution_ppm_avg"] = df[tk_cols].mean(axis=1)

elements = ["au", "ag", "cu", "zn"]
labels = {
    "au": "AU Average",
    "ag": "AG Average",
    "cu": "CU Average",
    "zn": "ZN Average",
}
colours = {
    "au": "#4F6DFF",
    "ag": "#F05A28",
    "cu": "#20B486",
    "zn": "#A66BFF",
}
fill_colours = {
    "au": "rgba(79,109,255,0.10)",
    "ag": "rgba(240,90,40,0.10)",
    "cu": "rgba(32,180,134,0.10)",
    "zn": "rgba(166,107,255,0.10)",
}

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=[labels[e] for e in elements],
)

for i, element in enumerate(elements, start=1):
    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df[f"{element}_solution_ppm_avg"],
            mode="lines",
            name=labels[element],
            line=dict(
                color=colours[element],
                width=2.5,
                shape="linear",
            ),
            fill="tozeroy",
            fillcolor=fill_colours[element],
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Date: %{x|%d %b %Y}<br>"
                "Value: %{y:,.2f} ppm"
                "<extra></extra>"
            ),
            legendgroup=element,
            showlegend=True,
        ),
        row=i,
        col=1,
    )

    fig.update_yaxes(
        title_text="ppm",
        row=i,
        col=1,
        showgrid=True,
        gridcolor="rgba(180, 190, 210, 0.22)",
        zeroline=False,
        showline=True,
        linewidth=1,
        linecolor="rgba(80, 90, 110, 0.45)",
        ticks="outside",
        tickfont=dict(size=11),
        title_font=dict(size=12),
    )

for r in range(1, 5):
    fig.update_xaxes(
        row=r,
        col=1,
        showgrid=True,
        gridcolor="rgba(180, 190, 210, 0.15)",
        showline=True,
        linewidth=1,
        linecolor="rgba(80, 90, 110, 0.45)",
        ticks="outside",
        tickfont=dict(size=11),
        type="date",
    )


fig.update_layout(
    title=dict(
        text="Tank Solution Averages Over Time",
        x=0.03,
        xanchor="left",
        font=dict(size=24),
    ),
    height=980,
    width=1350,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=90, r=120, t=90, b=70),
    legend=dict(
        title="Series",
        orientation="v",
        y=1.02,
        x=1.01,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(120,120,120,0.2)",
        borderwidth=1,
        font=dict(size=12),
    ),
)

# refine subplot title styling
if fig.layout.annotations:
    for annotation in list(fig.layout.annotations):
        annotation.font = dict(size=16, color="#2a3f5f")
        annotation.x = 0.5

fig.show()

In [ ]:
# Convert to mass per tonne
df_daily["cn_complexed_kgpt"] = df_daily["cn_complexed_gpl"] / 1000

# Multiply by throughput
df_daily["cn_complexed_tpd"] = df_daily["cn_complexed_kgpt"] * df_daily["throughput_tpd"] / 1000

# Specific NaCN Consumption (kg/t)
df_daily["cn_consumption_tpd_calc"] = (
    df_daily["specific_nacn_kgpt"] * df_daily["throughput_tpd"] / 1000
)

# Complexation index - proportion of cyanide that is complexed vs free in solution
df_daily["cn_complexation_index"] = (
    df_daily["cn_complexed_gpl"] / df_daily["wad_gpl_tk_8_s"]
)

In [ ]:
df_daily[["cu_feed_ppm", "cn_complexed_gpl", "nacn_consumption_tpd", "recovery_au_pct", "cn_complexation_index", "cn_consumption_tpd_calc", "cn_complexed_tpd"]].corr()

# Plot heatmap of correlations
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 8))
sns.heatmap(
	df_daily[["cu_feed_ppm", "cn_complexed_gpl", "nacn_consumption_tpd", "recovery_au_pct", "cn_complexation_index", "cn_consumption_tpd_calc", "cn_complexed_tpd"]].corr(),
	annot=True,
	cmap="coolwarm",
	fmt=".2f",
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
df = df_daily.copy()

# throughput_tpd                -> plant throughput (t/d)
# nacn_consumption_tpd       	-> reported NaCN consumption (t/d)
# specific_nacn_kgpt            -> specific NaCN consumption (kg/t)
# free_cn_ppm                   -> free cyanide concentration (ppm = mg/L)
# wad_cn_gpl                    -> WAD cyanide concentration (g/L as NaCN equivalent)
# cu_feed_ppm                   -> feed copper grade/content

required_cols = [
    "throughput_tpd",
    "nacn_consumption_tpd",
    "specific_nacn_kgpt",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [ ]:
# Calculate cyanide consumption from throughput and specific consumption
df["nacn_consumption_tpd_calc"] = (
    df["specific_nacn_kgpt"] * df["throughput_tpd"] / 1000
)

In [ ]:
# Compare calculated vs reported consumption
df["nacn_consumption_gap_tpd"] = (
    df["nacn_consumption_tpd"] - df["nacn_consumption_tpd_calc"]
)

df["nacn_consumption_gap_pct"] = np.where(
    df["nacn_consumption_tpd_calc"].abs() > 1e-9,
    100 * df["nacn_consumption_gap_tpd"] / df["nacn_consumption_tpd_calc"],
    np.nan,
)

In [ ]:
# Field mapping - adjust names to your dataset
free_cn_col = "free_cn_ppm"
wad_cn_col = "wad_cn_gpl"

if free_cn_col in df.columns:
    df["free_cn_gpl"] = pd.to_numeric(df[free_cn_col], errors="coerce") / 1000

if wad_cn_col in df.columns:
    df["wad_cn_gpl_clean"] = pd.to_numeric(df[wad_cn_col], errors="coerce")

In [ ]:
# =============================================================================
# WRANGLING + OUTLIER DETECTION
# =============================================================================

df = df_daily.copy()

# --- Ensure sorted ---
df = df.sort_values("date").reset_index(drop=True)


# -----------------------------------------------------------------------------
# OUTLIER DETECTION (IQR METHOD - ROBUST FOR PROCESS DATA)
# -----------------------------------------------------------------------------
def detect_outliers_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - k * iqr
    upper = q3 + k * iqr

    return (series < lower) | (series > upper), lower, upper


def winsorise_series(series, lower, upper):
    return series.clip(lower, upper)


# -----------------------------------------------------------------------------
# APPLY TO KEY PROCESS VARIABLES
# -----------------------------------------------------------------------------
cols_to_clean = [
    "throughput_tpd",
    "au_feed_gpt",
    "au_tail_gpt",
    "cu_feed_ppm",
    "cyanide_consumption_tpd",
    "specific_nacn_kgpt",
    "do_avg",
]

outlier_summary = []

for col in cols_to_clean:
    mask, lower, upper = detect_outliers_iqr(df[col])

    n_outliers = mask.sum()

    df[f"{col}_clean"] = winsorise_series(df[col], lower, upper)

    outlier_summary.append({
        "column": col,
        "outliers_detected": int(n_outliers),
        "lower_bound": lower,
        "upper_bound": upper
    })

outlier_summary_df = pd.DataFrame(outlier_summary)

print("\nOutlier Summary:")
print(outlier_summary_df)


# =============================================================================
# SMOOTHING (ROLLING MEDIAN – BETTER THAN MEAN FOR PROCESS DATA)
# =============================================================================
window = 7  # 7-day smoothing (adjustable)

for col in cols_to_clean:
    df[f"{col}_smooth"] = df[f"{col}_clean"].rolling(
        window=window, center=True, min_periods=1
    ).median()


# =============================================================================
# RESIDENCE TIME CALCULATION
# =============================================================================
# Assumption:
# - throughput_tpd is solids throughput (t/day)
# - slurry density assumed ~1.3 t/m3 (adjust if you get plant value)

SLURRY_DENSITY = 1.3  # t/m3 (assumption – refine later)

df["throughput_m3_per_day"] = df["throughput_tpd"] / SLURRY_DENSITY

df["residence_time_hours"] = (
    total_circuit_volume / df["throughput_m3_per_day"]
) * 24

# Clean + smooth version
mask, lower, upper = detect_outliers_iqr(df["residence_time_hours"])
df["residence_time_hours_clean"] = winsorise_series(
    df["residence_time_hours"], lower, upper
)

df["residence_time_hours_smooth"] = df["residence_time_hours_clean"].rolling(
    window=window, center=True, min_periods=1
).median()


# =============================================================================
# QA CHECKS
# =============================================================================
print("\nResidence Time Summary (hrs):")
print(df["residence_time_hours"].describe())

print("\nThroughput Summary (tpd):")
print(df["throughput_tpd"].describe())


# =============================================================================
# PLOTTING – THROUGHPUT (RAW vs CLEAN vs SMOOTH)
# =============================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    df["date"],
    df["throughput_tpd"],
    label="Raw",
    alpha=0.4,
)

plt.plot(
    df["date"],
    df["throughput_tpd_clean"],
    label="Outlier Treated",
    linewidth=2,
)

plt.plot(
    df["date"],
    df["throughput_tpd_smooth"],
    label="Smoothed (7d Median)",
    linewidth=2,
)

plt.title("Throughput – Raw vs Cleaned vs Smoothed")
plt.xlabel("Date")
plt.ylabel("Throughput (tpd)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# =============================================================================
# OPTIONAL: RESIDENCE TIME PLOT (VERY USEFUL FOR LEACH ANALYSIS)
# =============================================================================
plt.figure(figsize=(14, 6))

plt.plot(df["date"], df["residence_time_hours"], label="Raw", alpha=0.4)
plt.plot(df["date"], df["residence_time_hours_clean"], label="Cleaned")
plt.plot(df["date"], df["residence_time_hours_smooth"], label="Smoothed")

plt.title("Residence Time (hrs)")
plt.xlabel("Date")
plt.ylabel("Hours")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Make copy of cleaned + smoothed data for export or further analysis
df_final = df.copy()

# Drop outliers using IQR method for final dataset
for col in cols_to_clean:
	mask, lower, upper = detect_outliers_iqr(df_final[col])
	df_final[col] = winsorise_series(df_final[col], lower, upper)


# Plot distribution of final throughput
plt.figure(figsize=(10, 5))
plt.hist(df["throughput_tpd"], bins=30, color=def_colours[1], alpha=0.7)
plt.title("Distribution of Throughput (tpd)")
plt.xlabel("Throughput (tpd)")
plt.ylabel("Frequency")
plt.grid(True)

# Plot distribution of final throughput
plt.figure(figsize=(10, 5))
plt.hist(df_final["throughput_tpd"], bins=30, color=def_colours[0], alpha=0.7)
plt.title("Distribution of Final Throughput (tpd)")
plt.xlabel("Throughput (tpd)")
plt.ylabel("Frequency")
plt.grid(True)



## Imports and workbook location

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)



## Inspect workbook structure

In [ ]:
wb = openpyxl.load_workbook(file_path, data_only=False)

print("Sheets:")
for s in wb.sheetnames:
    ws = wb[s]
    print(f" - {s}: rows={ws.max_row}, cols={ws.max_column}")

## Helper functions for flattening and cleaning

### Preview top rows from each sheet

In [ ]:
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    print(f"\n--- {sheet_name} ---")
    for r in range(1, min(8, ws.max_row) + 1):
        row_vals = [ws.cell(r, c).value for c in range(1, min(ws.max_column, 20) + 1)]
        print(f"Row {r}: {row_vals}")

### Helper functions

In [ ]:
def fill_merged_cells_inplace(ws):
    """
    For merged ranges, copy the top-left value into all cells in the merged range.
    This is useful before exporting worksheet values into a DataFrame.
    """
    merged_ranges = list(ws.merged_cells.ranges)
    for merged_range in merged_ranges:
        min_col = merged_range.min_col
        min_row = merged_range.min_row
        max_col = merged_range.max_col
        max_row = merged_range.max_row
        
        top_left_value = ws.cell(min_row, min_col).value
        
        # Unmerge first so values can be written everywhere
        ws.unmerge_cells(str(merged_range))
        for row in range(min_row, max_row + 1):
            for col in range(min_col, max_col + 1):
                ws.cell(row, col).value = top_left_value


def worksheet_to_dataframe(ws):
    data = ws.values
    df = pd.DataFrame(data)
    return df


def make_unique(names):
    seen = {}
    out = []
    for name in names:
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}_{seen[name]}")
    return out


def flatten_multilevel_header(header_rows):
    """
    header_rows: list of equally-sized lists/Series
    Combines non-null pieces into a single column name.
    """
    combined = []
    for parts in zip(*header_rows):
        clean_parts = []
        for p in parts:
            if p is None:
                continue
            p = str(p).strip()
            if p == "" or p.lower() == "none":
                continue
            clean_parts.append(p)
        combined.append(" | ".join(clean_parts) if clean_parts else "unnamed")
    return make_unique(combined)


def convert_excel_time_to_string(x):
    if pd.isna(x):
        return np.nan
    if hasattr(x, "strftime"):
        try:
            return x.strftime("%H:%M:%S")
        except Exception:
            return str(x)
    return str(x)


def standardise_column_names(df):
    cols = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    df.columns = cols
    return df

## Load and clean the Operational Data sheet

In [ ]:
ws_op = wb["Operational Data"]

# Convert sheet to raw DataFrame
df_op_raw = worksheet_to_dataframe(ws_op)

# Rows 3 and 4 in Excel are the header rows -> zero-index 2 and 3 in pandas
header_rows = [
    df_op_raw.iloc[2].tolist(),
    df_op_raw.iloc[3].tolist(),
]

op_cols = flatten_multilevel_header(header_rows)

df_op = df_op_raw.iloc[4:].copy()
df_op.columns = op_cols
df_op = standardise_column_names(df_op)

print(df_op.head())
print(df_op.columns.tolist())

## Clean daily operational data types

In [ ]:
# Rename a few key columns for easier handling
rename_map = {
    "Date": "date",
    "Throughput | tms (t/d)": "throughput_tpd",
    "Feed Grade | Au (g/t)": "feed_au_gpt",
    "Feed Grade | Ag (g/t)": "feed_ag_gpt",
    "Feed Grade | Cu (g/t)": "feed_cu_gpt",
    "Tailings Grade | Au (g/t)": "tails_au_gpt",
    "Tailings Grade | Ag (g/t)": "tails_ag_gpt",
    "Cyanide Consumption | (t/d)": "nacn_tpd",
    "Specific NaCN Consumption | (kg/tms)": "nacn_kgpt",
    "Tailings Moisture | %": "tail_moisture_pct",
    "Dissolved Oxygen – Leach Tanks | AG1": "do_ag1",
    "Dissolved Oxygen – Leach Tanks | AG2": "do_ag2",
    "Dissolved Oxygen – Leach Tanks | AG3": "do_ag3",
    "Dissolved Oxygen – Leach Tanks | AG4": "do_ag4",
    "Dissolved Oxygen – Leach Tanks | AG5": "do_ag5",
    "Dissolved Oxygen – Leach Tanks | AG6": "do_ag6",
    "Dissolved Oxygen – Leach Tanks | AG7": "do_ag7",
    "Dissolved Oxygen – Leach Tanks | AG8": "do_ag8",
}
df_op = df_op.rename(columns=rename_map)

# Fix date column
df_op["date"] = pd.to_datetime(df_op["date"], errors="coerce")

# Convert all other columns except date to numeric
for c in df_op.columns:
    if c != "date":
        df_op[c] = pd.to_numeric(df_op[c], errors="coerce")

# Drop empty rows
df_op = df_op.dropna(subset=["date"]).reset_index(drop=True)

print(df_op.info())
df_op.head()

## Quick audit of Operational Data

In [ ]:
audit_op = pd.DataFrame({
    "column": df_op.columns,
    "dtype": [str(df_op[c].dtype) for c in df_op.columns],
    "non_null": [df_op[c].notna().sum() for c in df_op.columns],
    "nulls": [df_op[c].isna().sum() for c in df_op.columns],
    "min": [df_op[c].min() if c != "date" else df_op[c].min() for c in df_op.columns],
    "max": [df_op[c].max() if c != "date" else df_op[c].max() for c in df_op.columns],
})
audit_op

## Load and clean the Operational Data sheet

### Parse Operational Data

In [ ]:
ws_op = wb["Operational Data"]

# Convert sheet to raw DataFrame
df_op_raw = worksheet_to_dataframe(ws_op)

# Rows 3 and 4 in Excel are the header rows -> zero-index 2 and 3 in pandas
header_rows = [
    df_op_raw.iloc[2].tolist(),
    df_op_raw.iloc[3].tolist(),
]

op_cols = flatten_multilevel_header(header_rows)

df_op = df_op_raw.iloc[4:].copy()
df_op.columns = op_cols
df_op = standardise_column_names(df_op)

print(df_op.head())
print(df_op.columns.tolist())

### Clean daily operational data types

In [ ]:
# Rename a few key columns for easier handling
rename_map = {
    "Date": "date",
    "Throughput | tms (t/d)": "throughput_tpd",
    "Feed Grade | Au (g/t)": "feed_au_gpt",
    "Feed Grade | Ag (g/t)": "feed_ag_gpt",
    "Feed Grade | Cu (g/t)": "feed_cu_gpt",
    "Tailings Grade | Au (g/t)": "tails_au_gpt",
    "Tailings Grade | Ag (g/t)": "tails_ag_gpt",
    "Cyanide Consumption | (t/d)": "nacn_tpd",
    "Specific NaCN Consumption | (kg/tms)": "nacn_kgpt",
    "Tailings Moisture | %": "tail_moisture_pct",
    "Dissolved Oxygen – Leach Tanks | AG1": "do_ag1",
    "Dissolved Oxygen – Leach Tanks | AG2": "do_ag2",
    "Dissolved Oxygen – Leach Tanks | AG3": "do_ag3",
    "Dissolved Oxygen – Leach Tanks | AG4": "do_ag4",
    "Dissolved Oxygen – Leach Tanks | AG5": "do_ag5",
    "Dissolved Oxygen – Leach Tanks | AG6": "do_ag6",
    "Dissolved Oxygen – Leach Tanks | AG7": "do_ag7",
    "Dissolved Oxygen – Leach Tanks | AG8": "do_ag8",
}
df_op = df_op.rename(columns=rename_map)

# Fix date column
df_op["date"] = pd.to_datetime(df_op["date"], errors="coerce")

# Convert all other columns except date to numeric
for c in df_op.columns:
    if c != "date":
        df_op[c] = pd.to_numeric(df_op[c], errors="coerce")

# Drop empty rows
df_op = df_op.dropna(subset=["date"]).reset_index(drop=True)

print(df_op.info())
df_op.head()

### Quick audit of Operational Data

In [ ]:
audit_op = pd.DataFrame({
    "column": df_op.columns,
    "dtype": [str(df_op[c].dtype) for c in df_op.columns],
    "non_null": [df_op[c].notna().sum() for c in df_op.columns],
    "nulls": [df_op[c].isna().sum() for c in df_op.columns],
    "min": [df_op[c].min() if c != "date" else df_op[c].min() for c in df_op.columns],
    "max": [df_op[c].max() if c != "date" else df_op[c].max() for c in df_op.columns],
})
audit_op

### Date continuity check for operational data

In [ ]:
date_range = pd.date_range(df_op["date"].min(), df_op["date"].max(), freq="D")
missing_dates_op = sorted(set(date_range) - set(df_op["date"]))

print("Operational date range:", df_op["date"].min(), "to", df_op["date"].max())
print("Expected days:", len(date_range))
print("Actual days:", df_op["date"].nunique())
print("Missing days:", len(missing_dates_op))

missing_dates_op[:20]

## Load and clean the Leaching sheets

### Helper to parse one leaching sheet

In [ ]:
def parse_leaching_sheet(workbook, sheet_name):
    # Reload workbook fresh so merged cells are still intact when starting
    wb_local = openpyxl.load_workbook(file_path, data_only=False)
    ws = wb_local[sheet_name]
    
    # Fill merged cells so dates propagate down each block
    fill_merged_cells_inplace(ws)
    
    df_raw = worksheet_to_dataframe(ws)
    
    # First 2 rows are headers
    header_rows = [
        df_raw.iloc[0].tolist(),
        df_raw.iloc[1].tolist(),
    ]
    cols = flatten_multilevel_header(header_rows)
    
    df = df_raw.iloc[2:].copy()
    df.columns = cols
    df = standardise_column_names(df)
    
    # Rename first two columns
    rename_map = {
        df.columns[0]: "date",
        df.columns[1]: "time",
    }
    df = df.rename(columns=rename_map)
    
    # Date/time cleanup
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["time_str"] = df["time"].apply(convert_excel_time_to_string)
    
    # Combined timestamp if possible
    df["timestamp"] = pd.to_datetime(
        df["date"].dt.strftime("%Y-%m-%d") + " " + df["time_str"],
        errors="coerce"
    )
    
    # Convert measurement columns to numeric
    measure_cols = [c for c in df.columns if c not in ["date", "time", "time_str", "timestamp"]]
    for c in measure_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    
    # Drop rows with no date and no time and no measurements
    df = df.dropna(subset=["date", "timestamp"], how="all").reset_index(drop=True)
    
    return df

### Parse the leaching sheets

In [ ]:
df_l25 = parse_leaching_sheet(wb, "Leaching 2025")
df_l26 = parse_leaching_sheet(wb, "Leaching 2026")

print("Leaching 2025 shape:", df_l25.shape)
print("Leaching 2026 shape:", df_l26.shape)

display(df_l25.head())
display(df_l26.head())

### Inspect leaching columns

In [ ]:
print("Leaching 2025 columns:")
for c in df_l25.columns:
    print(c)

print("\nLeaching 2026 columns:")
for c in df_l26.columns:
    print(c)

## First reconciliation checks on the leach sheets

### Date coverage checks

In [ ]:
def sheet_date_audit(df, name):
    print(f"\n--- {name} ---")
    print("Min date:", df["date"].min())
    print("Max date:", df["date"].max())
    print("Unique dates:", df["date"].nunique())
    print("Rows:", len(df))
    print("Unique timestamps:", df["timestamp"].nunique())
    print("Null timestamps:", df["timestamp"].isna().sum())

sheet_date_audit(df_l25, "Leaching 2025")
sheet_date_audit(df_l26, "Leaching 2026")

### Daily expected sample count check

In [ ]:
def daily_sample_count(df, name):
    counts = (
        df.groupby("date")["timestamp"]
        .count()
        .rename("samples_per_day")
        .reset_index()
    )
    print(f"\n{name} sample count summary")
    print(counts["samples_per_day"].describe())
    return counts

l25_counts = daily_sample_count(df_l25, "Leaching 2025")
l26_counts = daily_sample_count(df_l26, "Leaching 2026")

display(l25_counts.head())
display(l26_counts.head())

### Missing dates in the leaching sheets

In [ ]:
def missing_dates(df):
    all_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    actual = set(df["date"].dropna().dt.normalize())
    missing = sorted(set(all_dates) - actual)
    return missing

missing_l25 = missing_dates(df_l25)
missing_l26 = missing_dates(df_l26)

print("Missing dates in Leaching 2025:", len(missing_l25))
print(missing_l25[:30])

print("\nMissing dates in Leaching 2026:", len(missing_l26))
print(missing_l26[:30])

## Daily aggregation of leach chemistry

### Aggregate leach chemistry to daily means

In [ ]:
def aggregate_daily_leach(df):
    numeric_cols = [c for c in df.columns if c not in ["date", "time", "time_str", "timestamp"]]
    
    daily = (
        df.groupby("date")[numeric_cols]
        .agg(["mean", "min", "max", "count"])
    )
    
    # flatten columns
    daily.columns = [f"{col[0]}__{col[1]}" for col in daily.columns]
    daily = daily.reset_index()
    return daily

daily_l25 = aggregate_daily_leach(df_l25)
daily_l26 = aggregate_daily_leach(df_l26)

display(daily_l25.head())
display(daily_l26.head())

### Combine leach daily data

In [ ]:
df_leach_daily = pd.concat([daily_l25, daily_l26], ignore_index=True)
df_leach_daily = df_leach_daily.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

print(df_leach_daily.shape)
display(df_leach_daily.head())
display(df_leach_daily.tail())

### Merge the daily datasets

In [ ]:
df_master = df_op.merge(df_leach_daily, on="date", how="left")

print("Master shape:", df_master.shape)
print("Operational days:", df_op["date"].nunique())
print("Merged days:", df_master["date"].nunique())

display(df_master.head())

### Identify overlap and gaps

In [ ]:
df_master["has_leach_data"] = df_master.filter(like="__mean").notna().any(axis=1)

coverage_summary = df_master["has_leach_data"].value_counts(dropna=False)
print(coverage_summary)

missing_leach_overlap = df_master.loc[~df_master["has_leach_data"], ["date"]]
missing_leach_overlap.head(30)

### Quick range checks for key fields

In [ ]:
range_checks = {
    "throughput_tpd": (1000, 30000),
    "feed_au_gpt": (0, 20),
    "feed_ag_gpt": (0, 1000),
    "feed_cu_gpt": (0, 10000),
    "nacn_tpd": (0, 200),
    "nacn_kgpt": (0, 20),
    "tail_moisture_pct": (0, 100),
    "do_ag1": (0, 20),
    "do_ag2": (0, 20),
    "do_ag3": (0, 20),
    "do_ag4": (0, 20),
    "do_ag5": (0, 20),
    "do_ag6": (0, 20),
    "do_ag7": (0, 20),
    "do_ag8": (0, 20),
}

flags = []

for col, (lo, hi) in range_checks.items():
    if col in df_master.columns:
        bad = df_master[(df_master[col].notna()) & ((df_master[col] < lo) | (df_master[col] > hi))]
        if len(bad) > 0:
            tmp = bad[["date", col]].copy()
            tmp["issue"] = f"{col} outside [{lo}, {hi}]"
            flags.append(tmp)

df_flags = pd.concat(flags, ignore_index=True) if flags else pd.DataFrame(columns=["date", "value", "issue"])
display(df_flags.head(100))

## First diagnostic plots for reconciliation

### Operational trends

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

df_master.plot(x="date", y="throughput_tpd", ax=axes[0], title="Throughput")
df_master.plot(x="date", y="nacn_tpd", ax=axes[1], title="NaCN Consumption (t/d)")
df_master.plot(x="date", y="nacn_kgpt", ax=axes[2], title="Specific NaCN Consumption (kg/tms)")

plt.tight_layout()
plt.show()

### Compare NaCN consumption with feed Cu

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.plot(df_master["date"], df_master["nacn_kgpt"], label="NaCN kg/tms")
ax1.set_ylabel("NaCN kg/tms")

ax2 = ax1.twinx()
ax2.plot(df_master["date"], df_master["feed_cu_gpt"], linestyle="--", label="Feed Cu (g/t)")
ax2.set_ylabel("Feed Cu (g/t)")

ax1.set_title("Specific NaCN Consumption vs Feed Cu")
plt.show()

### Find Cu-related leach columns

In [ ]:
cu_cols = [c for c in df_master.columns if "Cu ppm" in c]
wad_cols = [c for c in df_master.columns if "WAD" in c or "Wad" in c or "wad" in c]
free_cn_cols = [c for c in df_master.columns if "NaCN" in c or "CN" in c]

print("Cu columns:")
for c in cu_cols[:50]:
    print(c)

print("\nWAD/CN-related columns:")
for c in free_cn_cols[:80]:
    print(c)

### Plot using one solution Cu field if present

In [ ]:
# Example only - update after inspecting your real column names
example_cu_col = None

for c in cu_cols:
    if "__mean" in c:
        example_cu_col = c
        break

print("Using:", example_cu_col)

if example_cu_col:
    fig, ax1 = plt.subplots(figsize=(16, 6))
    ax1.plot(df_master["date"], df_master["nacn_kgpt"], label="NaCN kg/tms")
    ax1.set_ylabel("NaCN kg/tms")

    ax2 = ax1.twinx()
    ax2.plot(df_master["date"], df_master[example_cu_col], linestyle="--")
    ax2.set_ylabel("Solution Cu ppm")

    ax1.set_title(f"Specific NaCN Consumption vs {example_cu_col}")
    plt.show()

## Build a formal audit log table

### Audit log

In [ ]:
audit_log = []

audit_log.append({
    "check": "Workbook sheets identified",
    "result": ", ".join(wb.sheetnames),
    "status": "done"
})

audit_log.append({
    "check": "Operational data date range",
    "result": f"{df_op['date'].min()} to {df_op['date'].max()}",
    "status": "done"
})

audit_log.append({
    "check": "Leaching 2025 date range",
    "result": f"{df_l25['date'].min()} to {df_l25['date'].max()}",
    "status": "done"
})

audit_log.append({
    "check": "Leaching 2026 date range",
    "result": f"{df_l26['date'].min()} to {df_l26['date'].max()}",
    "status": "done"
})

audit_log.append({
    "check": "Operational data missing dates",
    "result": len(missing_dates_op),
    "status": "review"
})

audit_log.append({
    "check": "Leaching 2025 missing dates",
    "result": len(missing_l25),
    "status": "review"
})

audit_log.append({
    "check": "Leaching 2026 missing dates",
    "result": len(missing_l26),
    "status": "review"
})

audit_log.append({
    "check": "Operational/leach overlap days without leach daily data",
    "result": int((~df_master["has_leach_data"]).sum()),
    "status": "review"
})

df_audit_log = pd.DataFrame(audit_log)
df_audit_log

## Save clean outputs for the next stage

### Export cleaned data

In [ ]:
output_dir = Path("la_coipa_outputs")
output_dir.mkdir(exist_ok=True)

df_op.to_csv(output_dir / "operational_data_clean.csv", index=False)
df_l25.to_csv(output_dir / "leaching_2025_clean.csv", index=False)
df_l26.to_csv(output_dir / "leaching_2026_clean.csv", index=False)
df_leach_daily.to_csv(output_dir / "leaching_daily_aggregated.csv", index=False)
df_master.to_csv(output_dir / "la_coipa_master_daily.csv", index=False)
df_audit_log.to_csv(output_dir / "data_audit_log.csv", index=False)

print("Saved files to:", output_dir.resolve())

## Diagnostic plots, correlation review, lag analysis, and first-pass copper-risk segmentation

This section is designed to help you answer:

1. How strongly NaCN consumption tracks feed Cu,
2. Whether solution Cu is a better explanatory signal than feed Cu,
3. Whether WAD CN rises with Cu,
4. Whether free CN collapses or is actively maintained,
5. Whether there are distinct operating regimes or campaigns,

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

output_dir = Path("la_coipa_outputs")
df_master = pd.read_csv(output_dir / "la_coipa_master_daily.csv")
df_master["date"] = pd.to_datetime(df_master["date"], errors="coerce")

print(df_master.shape)
df_master.head()